# Module 2 Guided Practice — Demonstrations and Worked Reasoning for Retail Returns

### Problem statement

A retailer's support team struggles with tricky returns involving **delivery dates**, **item condition** and **missing receipts**. **Test how selecting relevant examples and adding worked reasoning help an LLM apply the returns policy correctly.**

### What you will do

You will run one controlled experiment. The model, policy, tickets and decoding settings stay fixed; only the **in-context demonstrations** change:

| Condition | What is in the prompt |
|---|---|
| A. Zero-shot | policy + ticket |
| B. Convenience few-shot | policy + the **first k** examples from the pool + ticket |
| C. Selected few-shot | policy + **k examples you choose** to cover the decision boundaries + ticket |
| D. Worked reasoning | same examples as C, but each shows its **worked policy checks**, and the model writes its checks before deciding |

All few-shot conditions use the same **k = 4**, so the number of demonstrations is not a confound.

### Learning objectives

- Build a zero-shot baseline for a policy-classification task.
- Select demonstrations intentionally, checking **label coverage** and **decision-boundary coverage** (Lesson 2: selection, label distribution, recency).
- Add worked reasoning to demonstrations and observe where it helps (chain-of-thought, Wei et al. 2022).
- Evaluate with accuracy, per-label precision / recall / F1, run-to-run stability and **token cost**.
- Explain at least one error using the policy and course concepts.

### Ground rules

- Run cells in order. Tasks 2–5 fit in one class session. Task 6 is the long run: start it once, leave it, and bring the saved results back.
- Debug on **2 development tickets** first, then the full development set.
- Run the **evaluation set once**, after your prompts are final. Do not tune on it.
- Every run is saved to `results/` with a timestamp. Nothing is overwritten.


# Task 1 — Set up your environment (do this before class)

1. Put this notebook in a project folder, for example `returns-lab/`.
2. In that folder, create a virtual environment with **Python 3.12** and install the packages:
   ```
   uv venv --python 3.12
   uv pip install -r requirements.txt
   ```
   `requirements.txt` must include **`openai>=1.66`** (this notebook uses the Responses API) and **`pandas`**.
3. Create a file named `.env` in the same folder with one line:
   ```
   OPENAI_API_KEY=sk-...
   ```
   Never paste your key into the notebook or share it.
4. In VS Code or JupyterLab, select the `.venv` kernel.

**Quick check:** run the first code cell in Task 2. If it prints the model name, you are ready.

| Error | Likely fix |
|---|---|
| `ModuleNotFoundError` | Wrong kernel selected; pick `.venv` |
| `'OpenAI' object has no attribute 'responses'` | Old `openai` package; run `uv pip install -U "openai>=1.66"` |
| `AuthenticationError` / 401 | Check the key in `.env`, then restart the kernel |

# Task 2 — Load the policy, data, and model helper

The default model is `gpt-4.1-mini`. A model that is too strong can solve this task zero-shot, which would hide the differences you are here to measure; if your zero-shot baseline is already perfect on the development set, tell your instructor. Temperature is set to 0 so that differences between conditions come from the prompt, not sampling (Lesson 1).

Before running, make sure the `.env` file from Task 1 is in the project root and the `.venv` kernel is selected.

In [ ]:
# %pip install -q "openai>=1.66" pandas python-dotenv   # uncomment in a fresh environment

import os, re, json, time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)   # reads OPENAI_API_KEY from .env; override=True picks up a changed key
client = OpenAI()

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")   # use a non-reasoning model: o-series / gpt-5 models reject temperature
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 400          # room for worked checks in Condition D
K = 4                            # demonstrations per few-shot prompt (fixed across B, C, D)

# USD per 1M tokens for MODEL_NAME. Check current pricing and update before the lab.
PRICE_PER_1M = {"input": 0.40, "output": 1.60}   # gpt-4.1-mini list price

LABELS = ["APPROVE_RETURN", "DENY_RETURN", "REVIEW_REQUIRED"]
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

def prompt_model(prompt, retries=3):
    """Send one prompt; return (text, input_tokens, output_tokens). Retries on rate limits or network errors."""
    for attempt in range(retries):
        try:
            r = client.responses.create(
                model=MODEL_NAME,
                input=prompt,
                temperature=TEMPERATURE,
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )
            return r.output_text.strip(), r.usage.input_tokens, r.usage.output_tokens
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  API error ({type(e).__name__}), retrying in {wait}s...")
            time.sleep(wait)

print("Model:", MODEL_NAME, "| temperature:", TEMPERATURE, "| k =", K)

### The return policy - this is a fictional return policy for this lab. 

Read it closely. Three details cause most errors:

- The window is counted from the **delivery date**, not the order date, and the last eligible day is included (day 14 is in, day 15 is out).
- **Customer-caused damage is not a defect**, even when the customer calls it one.
- A **bank or card statement alone is not proof of purchase**.

In [ ]:
RETURN_POLICY = """
RETAIL RETURN POLICY

1. Return window
   - Count calendar days from the DELIVERY date (not the order date) to the date of the request.
   - Standard returns: the request must be made within 14 days after delivery (day 14 is eligible, day 15 is not).
   - Defective or damaged-on-arrival items: within 30 days after delivery (day 30 is eligible, day 31 is not).

2. Item condition (standard returns)
   - The item must be unused, or opened only for inspection, and remain resalable.
   - Items that were used, worn, washed, assembled and used, or damaged by the customer are not eligible.

3. Proof of purchase
   - Acceptable: paper receipt, order number, order confirmation email, or order record in the customer's account.
   - A bank or card statement alone is NOT acceptable proof.
   - If a request is otherwise eligible but no acceptable proof is available, return REVIEW_REQUIRED.

4. Defective or damaged-on-arrival items
   - Covers manufacturing defects under normal use and items that arrived damaged.
   - Damage caused by the customer (drops, spills, misuse, pets) is NOT a defect.
   - The item may have been opened or used. Proof of purchase is still required.

5. Decision order
   a. If the request is outside the applicable window -> DENY_RETURN.
   b. If the item is not defective and its condition makes it ineligible -> DENY_RETURN.
   c. If the request is otherwise eligible but proof cannot be verified -> REVIEW_REQUIRED.
   d. Otherwise -> APPROVE_RETURN.
""".strip()

print(RETURN_POLICY)

### The data

Each ticket has a ticket date, order date, carrier delivery date and the customer's own words. The model must do the date arithmetic itself.

- `demonstration_pool` plays the role of the **training set**: you sample demonstrations from it. Each entry is tagged with the decision boundary it illustrates (`covers`) and a set of worked policy checks (`checks`).
- `dev_cases` are for building and debugging prompts.
- `eval_cases` are held out for the final comparison.

In [ ]:
demonstration_pool = [{'id': 'P1',
  'ticket': '2026-03-06',
  'ordered': '2026-02-26',
  'delivered': '2026-03-02',
  'msg': 'Bought the wrong size of these running shoes. Never worn, still in the box. Order #A1182.',
  'label': 'APPROVE_RETURN',
  'covers': ['easy_approve'],
  'checks': ['Window: delivered 2026-03-02, request 2026-03-06 -> 4 days after delivery. Standard window (14 '
             'days) -> inside.',
             'Condition: never worn, in the box -> resalable.',
             'Proof: order number -> acceptable.',
             'Defect exception: none reported -> not needed.']},
 {'id': 'P2',
  'ticket': '2026-04-15',
  'ordered': '2026-04-06',
  'delivered': '2026-04-10',
  'msg': "Changed my mind about this blender, it's still sealed. I have the paper receipt.",
  'label': 'APPROVE_RETURN',
  'covers': ['easy_approve'],
  'checks': ['Window: delivered 2026-04-10, request 2026-04-15 -> 5 days. Standard window -> inside.',
             'Condition: sealed -> resalable.',
             'Proof: paper receipt -> acceptable.',
             'Defect exception: none reported -> not needed.']},
 {'id': 'P3',
  'ticket': '2026-02-20',
  'ordered': '2025-12-29',
  'delivered': '2026-01-05',
  'msg': "Never got around to using this desk lamp, it's unopened. Order #L-5521.",
  'label': 'DENY_RETURN',
  'covers': ['window_deny_obvious'],
  'checks': ['Window: delivered 2026-01-05, request 2026-02-20 -> 46 days. No defect, so the 14-day window '
             'applies -> outside.',
             'Condition: unopened, but the window already makes it ineligible.',
             'Proof: order number (not decisive).',
             'Defect exception: none reported -> does not apply.']},
 {'id': 'P4',
  'ticket': '2026-05-09',
  'ordered': '2026-04-28',
  'delivered': '2026-05-01',
  'msg': "I opened the box to look at the throw pillow but the color is off from the photos. Didn't use it. "
         'I still have the confirmation email.',
  'label': 'APPROVE_RETURN',
  'covers': ['inspection_ok'],
  'checks': ['Window: delivered 2026-05-01, request 2026-05-09 -> 8 days. Standard window -> inside.',
             'Condition: opened only to inspect, unused -> resalable.',
             'Proof: order confirmation email -> acceptable.',
             'Defect exception: none reported -> not needed.']},
 {'id': 'P5',
  'ticket': '2026-03-16',
  'ordered': '2026-02-20',
  'delivered': '2026-03-03',
  'msg': "Shipping took forever and I no longer need this printer. It's unopened. Order #P-7730.",
  'label': 'APPROVE_RETURN',
  'covers': ['order_vs_delivery_date'],
  'checks': ['Window: count from DELIVERY, not order. Delivered 2026-03-03, request 2026-03-16 -> 13 days '
             '(24 days from the order date, which does not matter). Standard window -> inside.',
             'Condition: unopened -> resalable.',
             'Proof: order number -> acceptable.',
             'Defect exception: none reported -> not needed.']},
 {'id': 'P6',
  'ticket': '2026-06-16',
  'ordered': '2026-05-27',
  'delivered': '2026-06-01',
  'msg': "These headphones are unopened, just didn't need them after all. Receipt attached.",
  'label': 'DENY_RETURN',
  'covers': ['standard_window_boundary'],
  'checks': ['Window: delivered 2026-06-01, request 2026-06-16 -> 15 days. Standard window ends at day 14 -> '
             'outside by one day.',
             'Condition: unopened, but the window already makes it ineligible.',
             'Proof: receipt (not decisive).',
             'Defect exception: none reported -> does not apply.']},
 {'id': 'P7',
  'ticket': '2026-03-14',
  'ordered': '2026-03-05',
  'delivered': '2026-03-10',
  'msg': "Wore this jacket to two events and washed it once. It just isn't my style. Receipt is in the bag.",
  'label': 'DENY_RETURN',
  'covers': ['condition_used'],
  'checks': ['Window: delivered 2026-03-10, request 2026-03-14 -> 4 days. Standard window -> inside.',
             'Condition: worn and washed -> used, not resalable -> ineligible for a standard return.',
             'Proof: receipt (not decisive).',
             'Defect exception: no defect reported -> does not apply.']},
 {'id': 'P8',
  'ticket': '2026-07-09',
  'ordered': '2026-06-29',
  'delivered': '2026-07-02',
  'msg': 'The phone case cracked when I dropped my phone yesterday. A case should be sturdier than that, so '
         'I consider it defective. I have the order email.',
  'label': 'DENY_RETURN',
  'covers': ['customer_damage_vs_defect'],
  'checks': ['Window: delivered 2026-07-02, request 2026-07-09 -> 7 days. Standard window -> inside.',
             'Condition: damage was caused by the customer dropping it -> ineligible.',
             'Proof: order email (not decisive).',
             'Defect exception: customer-caused damage is not a manufacturing defect -> does not apply.']},
 {'id': 'P9',
  'ticket': '2026-04-12',
  'ordered': '2026-03-28',
  'delivered': '2026-04-03',
  'msg': "Unopened speaker. I don't have a receipt or order number, but I can send you my bank statement "
         'showing the charge.',
  'label': 'REVIEW_REQUIRED',
  'covers': ['missing_proof', 'bank_statement'],
  'checks': ['Window: delivered 2026-04-03, request 2026-04-12 -> 9 days. Standard window -> inside.',
             'Condition: unopened -> resalable.',
             'Proof: a bank statement alone is not acceptable proof -> cannot verify.',
             'Defect exception: none reported -> not needed. Otherwise eligible but unverifiable -> '
             'review.']},
 {'id': 'P10',
  'ticket': '2026-05-27',
  'ordered': '2026-04-30',
  'delivered': '2026-05-05',
  'msg': "My coffee grinder stopped turning on after about a week of normal use. It's in my account order "
         'history.',
  'label': 'APPROVE_RETURN',
  'covers': ['defect_approve'],
  'checks': ['Window: delivered 2026-05-05, request 2026-05-27 -> 22 days. Past 14, but a defect applies -> '
             '30-day window -> inside.',
             'Condition: used, which is allowed for a defective item.',
             'Proof: account order record -> acceptable.',
             'Defect exception: manufacturing defect under normal use -> applies.']},
 {'id': 'P11',
  'ticket': '2026-03-06',
  'ordered': '2026-01-27',
  'delivered': '2026-02-01',
  'msg': "The toaster's heating element died. I still have the receipt.",
  'label': 'DENY_RETURN',
  'covers': ['defect_window_boundary'],
  'checks': ['Window: delivered 2026-02-01, request 2026-03-06 -> 33 days (February 2026 has 28 days). '
             'Defect window is 30 days -> outside.',
             'Condition: not decisive; the window already makes it ineligible.',
             'Proof: receipt (not decisive).',
             'Defect exception: applies in principle, but its 30-day window has expired.']},
 {'id': 'P12',
  'ticket': '2026-06-29',
  'ordered': '2026-06-04',
  'delivered': '2026-06-10',
  'msg': "The vase arrived shattered inside the box. It was a gift from my aunt so I don't have a receipt or "
         'any order info.',
  'label': 'REVIEW_REQUIRED',
  'covers': ['defect_no_proof', 'missing_proof'],
  'checks': ['Window: delivered 2026-06-10, request 2026-06-29 -> 19 days. Damaged on arrival -> 30-day '
             'window -> inside.',
             'Condition: damaged on arrival, not by the customer.',
             'Proof: none available -> cannot verify.',
             'Defect exception: applies, but proof is still required -> review.']},
 {'id': 'P13',
  'ticket': '2026-08-10',
  'ordered': '2026-07-30',
  'delivered': '2026-08-03',
  'msg': "I've used these earbuds every day and my dog chewed one of the tips. No receipt, sorry.",
  'label': 'DENY_RETURN',
  'covers': ['decision_order_condition_before_proof'],
  'checks': ['Window: delivered 2026-08-03, request 2026-08-10 -> 7 days. Standard window -> inside.',
             "Condition: used daily and damaged by the customer's dog -> ineligible.",
             'Proof: missing, but the request is already ineligible on condition -> no review needed.',
             'Defect exception: no defect -> does not apply.']},
 {'id': 'P14',
  'ticket': '2026-02-02',
  'ordered': '2026-01-06',
  'delivered': '2026-01-10',
  'msg': 'Still-sealed board game, but I lost the receipt.',
  'label': 'DENY_RETURN',
  'covers': ['decision_order_window_before_proof'],
  'checks': ['Window: delivered 2026-01-10, request 2026-02-02 -> 23 days. No defect -> 14-day window -> '
             'outside.',
             'Condition: sealed, but the window already makes it ineligible.',
             'Proof: missing, but the request is already ineligible on the window -> no review needed.',
             'Defect exception: no defect -> does not apply.']}]

In [ ]:
# Development tickets: DEV1-DEV12. Evaluation tickets: E01-E30.
dev_cases = [{'id': 'DEV1',
  'ticket': '2026-03-22',
  'ordered': '2026-03-01',
  'delivered': '2026-03-09',
  'msg': "Package took a while. Monitor is still sealed, I just don't need it anymore. Confirmation email "
         'attached.',
  'label': 'APPROVE_RETURN'},
 {'id': 'DEV2',
  'ticket': '2026-05-05',
  'ordered': '2026-04-15',
  'delivered': '2026-04-20',
  'msg': "Opened the box to check the sheets' thread count, they're not what I expected. Have the receipt.",
  'label': 'DENY_RETURN'},
 {'id': 'DEV3',
  'ticket': '2026-06-11',
  'ordered': '2026-05-30',
  'delivered': '2026-06-03',
  'msg': "Used this vacuum for a week. It works fine, it's just too loud for our apartment. Order #V-3310.",
  'label': 'DENY_RETURN'},
 {'id': 'DEV4',
  'ticket': '2026-07-20',
  'ordered': '2026-07-10',
  'delivered': '2026-07-14',
  'msg': 'Never opened this. I checked out as a guest and deleted the email, so no order number.',
  'label': 'REVIEW_REQUIRED'},
 {'id': 'DEV5',
  'ticket': '2026-05-26',
  'ordered': '2026-04-28',
  'delivered': '2026-05-02',
  'msg': 'The kettle has leaked from the base seam since the first time I used it. Receipt attached.',
  'label': 'APPROVE_RETURN'},
 {'id': 'DEV6',
  'ticket': '2026-08-09',
  'ordered': '2026-07-28',
  'delivered': '2026-08-01',
  'msg': "Dropped the tablet off the couch and the screen cracked. Screens shouldn't crack that easily, I "
         "think it's a defect. Receipt available.",
  'label': 'DENY_RETURN'},
 {'id': 'DEV7',
  'ticket': '2026-04-15',
  'ordered': '2026-03-25',
  'delivered': '2026-04-01',
  'msg': 'Still sealed yoga mat. I changed my mind. Order #Y-4410.',
  'label': 'APPROVE_RETURN'},
 {'id': 'DEV8',
  'ticket': '2026-06-25',
  'ordered': '2026-06-04',
  'delivered': '2026-06-10',
  'msg': "Unopened desk lamp, we don't need it after all. Receipt attached.",
  'label': 'DENY_RETURN'},
 {'id': 'DEV9',
  'ticket': '2026-05-08',
  'ordered': '2026-04-27',
  'delivered': '2026-05-02',
  'msg': 'Washed this wool sweater once and it shrunk. I have the receipt.',
  'label': 'DENY_RETURN'},
 {'id': 'DEV10',
  'ticket': '2026-07-09',
  'ordered': '2026-06-26',
  'delivered': '2026-07-01',
  'msg': 'Sealed cast-iron skillet. I can show the card statement, but I have no receipt or order number.',
  'label': 'REVIEW_REQUIRED'},
 {'id': 'DEV11',
  'ticket': '2026-03-31',
  'ordered': '2026-02-22',
  'delivered': '2026-03-01',
  'msg': "The stand mixer motor stopped during normal use. It's in my account order history.",
  'label': 'APPROVE_RETURN'},
 {'id': 'DEV12',
  'ticket': '2026-02-03',
  'ordered': '2026-01-06',
  'delivered': '2026-01-12',
  'msg': 'Still-sealed chopping board, but I lost the receipt and have no order number.',
  'label': 'DENY_RETURN'}]

eval_cases = [{'id': 'E01',
  'ticket': '2026-03-30',
  'ordered': '2026-03-12',
  'delivered': '2026-03-16',
  'msg': 'Unopened air purifier, we bought two by accident. Order #H-9087.',
  'label': 'APPROVE_RETURN'},
 {'id': 'E02',
  'ticket': '2026-02-20',
  'ordered': '2026-02-02',
  'delivered': '2026-02-05',
  'msg': "I only opened the packaging to check the size of the frame. It won't fit the wall. Receipt "
         'included.',
  'label': 'DENY_RETURN'},
 {'id': 'E03',
  'ticket': '2026-09-04',
  'ordered': '2026-08-28',
  'delivered': '2026-09-01',
  'msg': "We laid the rug out in the living room for a few days and the puppy had an accident on it. We'd "
         'like to return it. Order #R-2204.',
  'label': 'DENY_RETURN'},
 {'id': 'E04',
  'ticket': '2026-04-13',
  'ordered': '2026-04-02',
  'delivered': '2026-04-07',
  'msg': 'The drone is still factory sealed. I paid with my credit card and can show the card statement, but '
         'I never got a receipt or order number.',
  'label': 'REVIEW_REQUIRED'},
 {'id': 'E05',
  'ticket': '2026-07-15',
  'ordered': '2026-06-15',
  'delivered': '2026-06-20',
  'msg': "On the third use the stand mixer's motor started smoking and smelling burnt. Order confirmation "
         'email attached.',
  'label': 'APPROVE_RETURN'},
 {'id': 'E06',
  'ticket': '2026-04-03',
  'ordered': '2026-02-25',
  'delivered': '2026-03-02',
  'msg': "The smartwatch won't hold a charge anymore, clearly a faulty battery. I have the receipt.",
  'label': 'DENY_RETURN'},
 {'id': 'E07',
  'ticket': '2026-06-02',
  'ordered': '2026-05-07',
  'delivered': '2026-05-11',
  'msg': 'The bookshelf arrived with a split side panel, I only just got around to reporting it. My partner '
         "ordered it on their account and I can't get the order details.",
  'label': 'REVIEW_REQUIRED'},
 {'id': 'E08',
  'ticket': '2026-08-20',
  'ordered': '2026-08-08',
  'delivered': '2026-08-12',
  'msg': 'The water bottle lid leaks now after I ran it through the dishwasher (I know the label says hand '
         'wash). Order #W-1150.',
  'label': 'DENY_RETURN'},
 {'id': 'E09',
  'ticket': '2026-07-09',
  'ordered': '2026-06-26',
  'delivered': '2026-07-01',
  'msg': 'Opened the box to make sure all the parts of the crib were there, never assembled it. We found a '
         "different one. It's in my account order history.",
  'label': 'APPROVE_RETURN'},
 {'id': 'E10',
  'ticket': '2026-02-19',
  'ordered': '2026-01-15',
  'delivered': '2026-01-20',
  'msg': 'Left earcup of the headphones went silent, no drops or damage. Receipt attached.',
  'label': 'APPROVE_RETURN'},
 {'id': 'E11',
  'ticket': '2026-04-28',
  'ordered': '2026-03-27',
  'delivered': '2026-04-01',
  'msg': "Unopened, nothing wrong with it, just don't want it. Lost the receipt though.",
  'label': 'DENY_RETURN'},
 {'id': 'E12',
  'ticket': '2026-09-10',
  'ordered': '2026-09-01',
  'delivered': '2026-09-05',
  'msg': "Used the air fryer twice, works fine, we just want a bigger one. I don't have a receipt.",
  'label': 'DENY_RETURN'},
 {'id': 'E13',
  'ticket': '2026-08-15',
  'ordered': '2026-07-26',
  'delivered': '2026-08-01',
  'msg': 'Opened the box to check the towel color against the photos, never used it. Confirmation email attached.',
  'label': 'APPROVE_RETURN'},
 {'id': 'E14',
  'ticket': '2026-09-22',
  'ordered': '2026-08-28',
  'delivered': '2026-09-12',
  'msg': "Shipping took a long time and I no longer need this fan. It's unopened. Order #F-2201.",
  'label': 'APPROVE_RETURN'},
 {'id': 'E15',
  'ticket': '2026-05-30',
  'ordered': '2026-05-04',
  'delivered': '2026-05-10',
  'msg': 'The iron started leaking water in the second week of normal use. Receipt attached.',
  'label': 'APPROVE_RETURN'},
 {'id': 'E16',
  'ticket': '2026-10-05',
  'ordered': '2026-09-28',
  'delivered': '2026-10-02',
  'msg': "Sealed humidifier, I bought the wrong model. It's in my account order history.",
  'label': 'APPROVE_RETURN'},
 {'id': 'E17',
  'ticket': '2026-11-10',
  'ordered': '2026-10-26',
  'delivered': '2026-11-01',
  'msg': "I opened the carton to count the dinner plates, didn't use them. Order #D-908.",
  'label': 'APPROVE_RETURN'},
 {'id': 'E18',
  'ticket': '2026-04-24',
  'ordered': '2026-04-01',
  'delivered': '2026-04-06',
  'msg': 'The desk lamp flickers and then goes dark after a few nights of ordinary use. Order confirmation email attached.',
  'label': 'APPROVE_RETURN'},
 {'id': 'E19',
  'ticket': '2026-03-19',
  'ordered': '2026-02-26',
  'delivered': '2026-03-04',
  'msg': 'Unopened kettle, we already own one. Receipt attached.',
  'label': 'DENY_RETURN'},
 {'id': 'E20',
  'ticket': '2026-06-17',
  'ordered': '2026-05-25',
  'delivered': '2026-06-01',
  'msg': 'Opened the box to look at the picture frame, never hung it. Order #R-17.',
  'label': 'DENY_RETURN'},
 {'id': 'E21',
  'ticket': '2026-02-17',
  'ordered': '2026-01-02',
  'delivered': '2026-01-08',
  'msg': 'Still sealed throw blanket, I never got around to using it. Order #B-300.',
  'label': 'DENY_RETURN'},
 {'id': 'E22',
  'ticket': '2026-07-17',
  'ordered': '2026-07-08',
  'delivered': '2026-07-12',
  'msg': 'Wore these sneakers around the house for two days. Receipt is in the shoebox.',
  'label': 'DENY_RETURN'},
 {'id': 'E23',
  'ticket': '2026-08-09',
  'ordered': '2026-07-28',
  'delivered': '2026-08-03',
  'msg': "I spilled coffee on the keyboard and now some keys stick. I think that means it's defective. Confirmation email attached.",
  'label': 'DENY_RETURN'},
 {'id': 'E24',
  'ticket': '2026-09-08',
  'ordered': '2026-08-27',
  'delivered': '2026-09-02',
  'msg': 'My cat scratched the sofa arm on the first night. Order #S-55.',
  'label': 'DENY_RETURN'},
 {'id': 'E25',
  'ticket': '2026-03-04',
  'ordered': '2026-01-26',
  'delivered': '2026-02-01',
  'msg': 'The space heater stopped producing heat. Receipt attached.',
  'label': 'DENY_RETURN'},
 {'id': 'E26',
  'ticket': '2026-06-11',
  'ordered': '2026-05-15',
  'delivered': '2026-05-20',
  'msg': "Sealed board game. I lost the receipt and don't have an order number.",
  'label': 'DENY_RETURN'},
 {'id': 'E27',
  'ticket': '2026-10-14',
  'ordered': '2026-10-06',
  'delivered': '2026-10-10',
  'msg': "Used this frying pan all week and food sticks to it. I don't have a receipt.",
  'label': 'DENY_RETURN'},
 {'id': 'E28',
  'ticket': '2026-03-28',
  'ordered': '2026-03-15',
  'delivered': '2026-03-20',
  'msg': 'Unopened air fryer. I can show the card charge, but I never received a receipt or order number.',
  'label': 'REVIEW_REQUIRED'},
 {'id': 'E29',
  'ticket': '2026-06-20',
  'ordered': '2026-06-01',
  'delivered': '2026-06-05',
  'msg': "The mirror arrived cracked inside the box. It was a gift, so I don't have the order details.",
  'label': 'REVIEW_REQUIRED'},
 {'id': 'E30',
  'ticket': '2026-08-25',
  'ordered': '2026-08-12',
  'delivered': '2026-08-18',
  'msg': 'Opened the package to check the sheet size, never used them. I checked out as a guest and deleted the email, so there is no order number.',
  'label': 'REVIEW_REQUIRED'}]

def format_ticket(t):
    return (f"Ticket date: {t['ticket']}\n"
            f"Order placed: {t['ordered']}\n"
            f"Delivered (carrier scan): {t['delivered']}\n"
            f'Customer message: "{t["msg"]}"')

demo_by_id = {d["id"]: d for d in demonstration_pool}

def label_counts(rows):
    return pd.Series([r["label"] for r in rows]).value_counts().reindex(LABELS, fill_value=0)

display(pd.DataFrame({
    "pool": label_counts(demonstration_pool),
    "dev": label_counts(dev_cases),
    "eval": label_counts(eval_cases),
}))
display(pd.DataFrame([{"id": d["id"], "label": d["label"], "covers": ", ".join(d["covers"])}
                      for d in demonstration_pool]))

### Check your understanding (2 minutes, no code)

Work out `DEV1` by hand before the model does. How many days after **delivery** was the request? What would you get if you mistakenly counted from the order date? Which label should it get?

In [ ]:
print(format_ticket(dev_cases[0]))

# Task 3 — Establish a zero-shot baseline

The prompt contains only the policy, one ticket and the allowed labels. Answers are parsed strictly: a response counts only if it contains exactly one allowed label. Anything else is recorded as `INVALID`, which counts as wrong.

In [ ]:
LABEL_RE = re.compile(r"\b(APPROVE_RETURN|DENY_RETURN|REVIEW_REQUIRED)\b")
DECISION_RE = re.compile(r"DECISION:\s*\**\s*(APPROVE_RETURN|DENY_RETURN|REVIEW_REQUIRED)")

def parse_label_only(text):
    hits = set(LABEL_RE.findall((text or "").upper()))
    return hits.pop() if len(hits) == 1 else "INVALID"

def parse_decision_line(text):
    """For worked reasoning: the checks may mention several labels, so only the final Decision line counts."""
    hits = DECISION_RE.findall((text or "").upper())
    return hits[-1] if hits else "INVALID"

ANSWER_LABEL_ONLY = """Choose exactly one label: APPROVE_RETURN, DENY_RETURN, REVIEW_REQUIRED.
Return the label only."""

def build_zero_shot_prompt(ticket):
    return f"""You are a retail returns assistant. Apply the policy exactly as written.

{RETURN_POLICY}

TICKET:
{format_ticket(ticket)}

{ANSWER_LABEL_ONLY}"""

print(build_zero_shot_prompt(dev_cases[0]))

In [ ]:
def run_experiment(cases, prompt_builder, parser, condition, split, repeats=1, show_raw=False):
    """Run every case `repeats` times, save the run to results/, and return a DataFrame."""
    rows = []
    for rep in range(repeats):
        for t in cases:
            raw, tin, tout = prompt_model(prompt_builder(t))
            pred = parser(raw)
            rows.append(dict(condition=condition, split=split, repeat=rep, id=t["id"],
                             expected=t["label"], predicted=pred, correct=pred == t["label"],
                             input_tokens=tin, output_tokens=tout, raw_response=raw))
            if show_raw:
                print(f"--- {t['id']} (repeat {rep}) -> {pred}\n{raw}\n")
    df = pd.DataFrame(rows)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    slug = re.sub(r"[^a-z0-9]+", "-", condition.lower()).strip("-")
    df.to_json(RESULTS_DIR / f"{split}_{slug}_{stamp}.json", orient="records", indent=2)
    return df

# Step 1: debug on two tickets only.
zero_dbg = run_experiment(dev_cases[:2], build_zero_shot_prompt, parse_label_only,
                          "A. Zero-shot", "debug", show_raw=True)

In [ ]:
# Step 2: once the two-ticket run works, run the full development set.
zero_dev = run_experiment(dev_cases, build_zero_shot_prompt, parse_label_only, "A. Zero-shot", "dev")
display(zero_dev[["id", "expected", "predicted", "correct"]])

In [ ]:
def accuracy_pct(df):
    """Percent of rows where predicted == expected."""
    n = len(df)
    n_correct = int(df["correct"].sum())
    return 100 * n_correct / n if n else float("nan"), n_correct, n

pct, n_correct, n = accuracy_pct(zero_dev)
print(f"Zero-shot baseline accuracy: {pct:.1f}% ({n_correct}/{n})")

If the baseline is already perfect on the development set, the task is too easy for this model and the comparison will not tell you much. Tell your instructor before switching models, because every condition must use the same one.

# Task 4 — Compare convenience and selected demonstrations

### Condition B: convenience

The quickest way to add examples is to take the first k from the pool. That is what Condition B does.

### Condition C: your selection

Pick **k = 4** demonstrations yourself. Use `describe_selection()` to check two things before you run anything:

1. **Label coverage.** A label that never appears in the demonstrations is hard for the model to produce. Compare against the development label distribution (Lesson 2: majority-label bias).
2. **Boundary coverage.** Does the set include the places where this policy is easy to misapply: the window counted from the wrong date, day 14 versus day 15, customer damage described as a defect, a bank statement offered as proof, and the decision order when signals conflict?

With only four slots you cannot cover everything, so decide which boundaries matter most and write your rule down. Also consider **order**: the last demonstration sits closest to the ticket (recency bias).

In [ ]:
IMPORTANT_BOUNDARIES = [
    "order_vs_delivery_date", "standard_window_boundary", "defect_window_boundary",
    "condition_used", "customer_damage_vs_defect", "missing_proof", "bank_statement",
    "defect_approve", "defect_no_proof",
    "decision_order_window_before_proof", "decision_order_condition_before_proof",
]

def describe_selection(ids, name="selection"):
    assert len(ids) == K, f"Use exactly K={K} demonstrations (got {len(ids)})."
    demos = [demo_by_id[i] for i in ids]
    covered = sorted({c for d in demos for c in d["covers"]} & set(IMPORTANT_BOUNDARIES))
    missing = [b for b in IMPORTANT_BOUNDARIES if b not in covered]
    print(f"{name}: {ids}")
    print("  label order :", " -> ".join(d["label"] for d in demos))
    print("  label counts:", {k: int(v) for k, v in label_counts(demos).items()})
    print("  dev counts  :", {k: int(v) for k, v in label_counts(dev_cases).items()})
    print(f"  boundaries covered ({len(covered)}/{len(IMPORTANT_BOUNDARIES)}):", covered)
    print("  not covered :", missing, "\n")

CONVENIENCE_DEMO_IDS = [d["id"] for d in demonstration_pool[:K]]
describe_selection(CONVENIENCE_DEMO_IDS, "B. Convenience")

In [ ]:
# TODO: choose exactly K demonstration ids from the pool, in the order they should appear.
# Note: until you fill this in, describe_selection() stops with an AssertionError. That is expected.
SELECTED_DEMO_IDS = []

# TODO: one sentence describing your selection rule.
SELECTION_RULE = ""

describe_selection(SELECTED_DEMO_IDS, "C. Selected")


In [ ]:
def format_demo_simple(d):
    return f"TICKET:\n{format_ticket(d)}\nDecision: {d['label']}"

def build_few_shot_prompt(ticket, demo_ids):
    examples = "\n\n---\n\n".join(format_demo_simple(demo_by_id[i]) for i in demo_ids)
    return f"""You are a retail returns assistant. Apply the policy exactly as written.

{RETURN_POLICY}

EXAMPLES:

{examples}

---

TICKET:
{format_ticket(ticket)}

{ANSWER_LABEL_ONLY}"""

convenience_prompt = lambda t: build_few_shot_prompt(t, CONVENIENCE_DEMO_IDS)
selected_prompt = lambda t: build_few_shot_prompt(t, SELECTED_DEMO_IDS)

conv_dev = run_experiment(dev_cases, convenience_prompt, parse_label_only, "B. Convenience few-shot", "dev")
sel_dev = run_experiment(dev_cases, selected_prompt, parse_label_only, "C. Selected few-shot", "dev")

def compare_by_case(*dfs):
    """One row per ticket, one column per condition. 1 means the label matched; 0 means it did not."""
    df = pd.concat(dfs, ignore_index=True)
    return df.pivot_table(index=["id", "expected"], columns="condition", values="correct",
                          aggfunc="mean", sort=False).round(2)

display(compare_by_case(zero_dev, conv_dev, sel_dev))


In the table above, **1** means the model's label matched the gold label on that ticket. **0** means it did not.

The comparison is `predicted == expected`. `expected` is the gold label stored on the ticket. `predicted` is the single label parsed from the reply. A reply with no label or more than one label is `INVALID` and counts as **0**.

Each prompt ran once on these 12 development tickets, so every cell is 1 or 0. The accuracy under the table is the number of 1s in that column divided by 12.


In [ ]:
def accuracy_by_condition(*dfs):
    """Share of 1s in each column: correct predictions divided by all predictions."""
    df = pd.concat(dfs, ignore_index=True)
    rows = []
    for cond, g in df.groupby("condition", sort=False):
        n, k = len(g), int(g["correct"].sum())
        rows.append({"condition": cond, "correct": k, "predictions": n,
                     "accuracy": f"{100 * k / n:.1f}%" if n else "n/a"})
    return pd.DataFrame(rows)

display(accuracy_by_condition(zero_dev, conv_dev, sel_dev))


**Selection checkpoint.** Find one development ticket whose result changed between B and C. Which demonstration do you think caused the change? Note it down; you will return to it in the discussion.

# Task 5 — Add worked reasoning to the demonstrations

Condition D keeps **the same four demonstrations in the same order** as Condition C. Two things change:

1. Each demonstration shows its worked policy checks: the day count, the condition, the proof and the defect exception, each written out before the decision.
2. The model is asked to write its own checks the same way and then finish with a single `Decision:` line.

This is few-shot chain-of-thought (Wei et al., 2022; see the CoT examples in Lesson 3). The day count is where written-out steps should matter most: the model has to subtract dates across month boundaries before it can apply any rule.

Because the prompt and the output format both change, D measures their combined effect. The optional extension at the end separates them.

In [ ]:
def format_demo_worked(d):
    checks = "\n".join(f"- {c}" for c in d["checks"])
    return f"TICKET:\n{format_ticket(d)}\nPolicy checks:\n{checks}\nDecision: {d['label']}"

ANSWER_WORKED = """Write your policy checks in the same format as the examples:
- Window: (count the days from the delivery date to the ticket date)
- Condition:
- Proof:
- Defect exception:
Then end with exactly one line: Decision: <APPROVE_RETURN | DENY_RETURN | REVIEW_REQUIRED>"""

def build_worked_prompt(ticket, demo_ids=None):
    demo_ids = demo_ids or SELECTED_DEMO_IDS
    examples = "\n\n---\n\n".join(format_demo_worked(demo_by_id[i]) for i in demo_ids)
    return f"""You are a retail returns assistant. Apply the policy exactly as written.

{RETURN_POLICY}

WORKED EXAMPLES:

{examples}

---

TICKET:
{format_ticket(ticket)}

{ANSWER_WORKED}"""

worked_dev = run_experiment(dev_cases, build_worked_prompt, parse_decision_line,
                            "D. Worked reasoning", "dev", show_raw=True)
display(compare_by_case(zero_dev, conv_dev, sel_dev, worked_dev))

Read one raw response above. Is the day count correct? If the decision is correct but a check is wrong, or the reverse, note it. Lesson 5 returns to whether written reasoning actually reflects how the model decided (faithfulness).

# Task 6 — Run the final evaluation and analyze errors

## Final evaluation (run once)

Freeze your selection and prompts, then run all four conditions on the evaluation set. Each condition runs **3 times**. At temperature 0 the outputs are usually, but not always, identical, so the repeats show how stable each condition is. Lesson 3 reported that few-shot exemplars reduced run-to-run variance in FLAN; check whether you see the same.

On `gpt-4.1-mini` this cell makes 360 calls (30 tickets, 4 prompts, 3 repeats) and should cost well under a dollar. It can take a while and prints nothing until each condition finishes.


In [ ]:
assert len(SELECTED_DEMO_IDS) == K and SELECTION_RULE, "Finish Task 4 before the final run."
REPEATS = 3

eval_runs = [
    run_experiment(eval_cases, build_zero_shot_prompt, parse_label_only, "A. Zero-shot", "eval", REPEATS),
    run_experiment(eval_cases, convenience_prompt, parse_label_only, "B. Convenience few-shot", "eval", REPEATS),
    run_experiment(eval_cases, selected_prompt, parse_label_only, "C. Selected few-shot", "eval", REPEATS),
    run_experiment(eval_cases, build_worked_prompt, parse_decision_line, "D. Worked reasoning", "eval", REPEATS),
]
all_eval = pd.concat(eval_runs, ignore_index=True)

In [ ]:
def macro_prf(df):
    """Per-label precision/recall/F1, macro-averaged over the three labels. INVALID counts as a miss."""
    p_list, r_list, f_list = [], [], []
    for lab in LABELS:
        tp = ((df.predicted == lab) & (df.expected == lab)).sum()
        fp = ((df.predicted == lab) & (df.expected != lab)).sum()
        fn = ((df.predicted != lab) & (df.expected == lab)).sum()
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        p_list.append(p); r_list.append(r); f_list.append(f)
    n = len(LABELS)
    return sum(p_list) / n, sum(r_list) / n, sum(f_list) / n

def summarize(df):
    rows = []
    for cond, g in df.groupby("condition", sort=False):
        acc_per_run = g.groupby("repeat")["correct"].mean()
        p, r, f = macro_prf(g)
        cost = (g.input_tokens.sum() * PRICE_PER_1M["input"] + g.output_tokens.sum() * PRICE_PER_1M["output"]) / 1e6
        rows.append(dict(
            condition=cond,
            accuracy=acc_per_run.mean(),
            accuracy_sd=acc_per_run.std(ddof=0),
            macro_precision=p, macro_recall=r, macro_f1=f,
            invalid=(g.predicted == "INVALID").sum(),
            avg_input_tokens=g.input_tokens.mean(),
            avg_output_tokens=g.output_tokens.mean(),
            usd_per_1000_tickets=cost / len(g) * 1000,
        ))
    return pd.DataFrame(rows).round(3)

summary = summarize(all_eval)
display(summary)

In [ ]:
# Confusion counts per condition: rows = expected, columns = predicted.
for cond, g in all_eval.groupby("condition", sort=False):
    print(cond)
    display(pd.crosstab(g.expected, g.predicted))

## Error analysis

The table below shows, for each evaluation ticket, the share of runs each condition got right. Look for tickets that **changed** between conditions, not just tickets that were wrong.

In [ ]:
by_case = compare_by_case(*eval_runs)
display(by_case)
display(accuracy_by_condition(*eval_runs))

errors = all_eval[~all_eval.correct].drop_duplicates(["condition", "id", "predicted"])
display(errors[["condition", "id", "expected", "predicted", "raw_response"]])


Pick **one** error and record:

| Field | Your answer |
|---|---|
| Ticket and condition | |
| Expected vs predicted | |
| Policy rule involved | window / date arithmetic, condition, proof, defect vs customer damage, or decision order |
| Likely cause | e.g. missing label in the demonstrations, recency or majority-label bias, no demonstration of this boundary, surface similarity to a misleading demonstration, arithmetic slip |
| One change you would test next | |

## Discussion questions

1. Did demonstrations help relative to zero-shot? Answer with your accuracy **and** macro-F1, and say whether the difference is larger than the run-to-run spread.
2. Compare B and C. Name one ticket that changed and the demonstration you think caused it. Was the gain from **label coverage**, **boundary coverage**, or both?
3. Where did worked reasoning help most: date arithmetic, conflicting signals (decision order), or defect-versus-damage? Where did it not help?
4. Worked reasoning used more output tokens. Using the `usd_per_1000_tickets` column, is the accuracy gain worth the cost at the scale of a real support team?
5. With 30 evaluation tickets, one ticket equals about 3 percentage points. What is the smallest difference you would trust, and what would you change to trust it more?
6. Write a selection rule you would give a colleague building demonstrations for a new policy, such as warranty claims.

Base your answers on your results. Few-shot and worked examples do not always perform better.


# Optional extensions (development set only)

Run these only on `dev_cases`. Do not re-run the evaluation set to tune them.

**E1. Order.** Keep your selected demonstrations and reverse them. Does the last demonstration pull predictions toward its label?

**E2. Reasoning without demonstrations.** Ask for the same written checks with no examples. Comparing this with Condition D separates the effect of the worked **examples** from the effect of asking the model to **write its reasoning**.

In [ ]:
# E1: order
reversed_ids = list(reversed(SELECTED_DEMO_IDS))
rev_dev = run_experiment(dev_cases, lambda t: build_few_shot_prompt(t, reversed_ids),
                         parse_label_only, "C'. Selected, reversed order", "dev")

# E2: written checks, no demonstrations
def build_zero_shot_checks_prompt(ticket):
    return f"""You are a retail returns assistant. Apply the policy exactly as written.

{RETURN_POLICY}

TICKET:
{format_ticket(ticket)}

{ANSWER_WORKED}"""

zs_checks_dev = run_experiment(dev_cases, build_zero_shot_checks_prompt, parse_decision_line,
                               "A'. Zero-shot + written checks", "dev")

display(compare_by_case(zero_dev, zs_checks_dev, sel_dev, rev_dev, worked_dev))

## Wrap-up

The experiment kept the model, policy, tickets and decoding fixed and changed only what was in the context:

**same model + same policy + same tickets → change the demonstrations → measure accuracy, F1, stability and cost.**

The graded assignment uses the same design at a larger scale, with multi-label outputs and a larger test set.